#### Import packages

In [1]:
import altair as alt
import pandas as pd

Import `publication-grids.csv` maintained by [Jane Pong](github.com/officeofjane)

Starting point [How to create a US Grid Cartogram using Altair](https://medium.com/@parham.motameni/how-to-create-a-us-grid-cartogram-using-altair-1800e8513639) posted by Parham Motameni.

In [2]:
url = 'https://gist.githubusercontent.com/officeofjane/2c3ed88c4be050d92765de912d71b7c4/raw/8abc5e19a5fac2ec63e81891d9bfdebe2d55f92c/publication-grids.csv'
df = pd.read_csv(url)

I want to see which publication includes Washington DC, so interested in those which have 51 grid cells.

In [3]:
df['publication'].value_counts()

publication
New York Times 1       51
NPR                    51
WNYC                   51
Wall Street Journal    51
National Journal       51
Guardian               51
Bloomberg 2            51
Axios                  51
Bloomberg 3            51
New York Times 2       50
Bloomberg 1            50
Propublica             50
538                    50
Marshall Project       50
Name: count, dtype: int64

In [4]:
publications_counts = df['publication'].value_counts()
type(publications_counts)

pandas.core.series.Series

In [5]:
# Get the publications with a count of 51
publications_counts = df['publication'].value_counts()

# Orders result
filtered_publications = publications_counts[publications_counts == 51].sort_index()

# Display the unique publications and convert the series to a DataFrame
unique_publications = filtered_publications.reset_index()
unique_publications.columns = ['publication', 'count']

unique_publications

,publication,count
0,Axios,51
1,Bloomberg 2,51
2,Bloomberg 3,51
3,Guardian,51
4,NPR,51
5,National Journal,51
6,New York Times 1,51
7,WNYC,51
8,Wall Street Journal,51


I like the grid for NPR - it includes DC and the placement of each grid seems best to me.

In [6]:
df = df[df['publication'] == 'NPR']

In [7]:
df.head()

,id,publication,state,code,col,row
151,152,NPR,Alaska,AK,2,1
152,153,NPR,Hawaii,HI,2,8
153,154,NPR,Washington,WA,3,3
154,155,NPR,Oregon,OR,3,4
155,156,NPR,California,CA,3,5


#### Trying with my own data

In [8]:
csv = pd.read_csv("data/50 states and abbreviations - Sheet1-2.csv")

In [9]:
csv.head()

,State,Abbreviation,duration,estimate,day_multiplier,days
0,Alabama,AL,months,9.0,30.0,270.0
1,Alaska,AK,days,NaN,2.0,2.0
2,Arizona,AZ,weeks,1.0,7.0,7.0
3,Arkansas,AR,hours,NaN,0.5,0.5
4,California,CA,months,7.0,30.0,210.0


Merge CSV with state grid dataframe on state abbreviations.

In [10]:
df = df.merge(csv, left_on='code', right_on='Abbreviation', how='left')

In [11]:
df.head()

,id,publication,state,code,col,row,State,Abbreviation,duration,estimate,day_multiplier,days
0,152,NPR,Alaska,AK,2,1,Alaska,AK,days,NaN,2.0,2.0
1,153,NPR,Hawaii,HI,2,8,Hawaii,HI,---,NaN,0.0,0.0
2,154,NPR,Washington,WA,3,3,Washington,WA,years,8.0,365.0,2920.0
3,155,NPR,Oregon,OR,3,4,Oregon,OR,weeks,1.5,7.0,10.5
4,156,NPR,California,CA,3,5,California,CA,months,7.0,30.0,210.0


The following is my first call to `Altair` to create a chart. I will use the `mark_bar()` method to create a bar chart.   
The `encode()` method is used to specify the x and y axes of the chart.   
The `x` axis will be the state, and the `y` axis will be the # of days. The `color` parameter is used to color the bars based on # of days.

In [12]:
heatmap = alt.Chart(df).mark_rect(stroke=None, cornerRadius=5).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None),
    color=alt.Color('days:Q', scale=alt.Scale(scheme='blues'), legend=alt.Legend(title='Days')),
    tooltip=['code', 'days']
).properties(
    width=600,
    height=400,
    title='State-Level Days Heatmap'
)


In [13]:
heatmap

alt.Chart(...)

In [14]:
text = heatmap.mark_text(baseline='middle', align='center', fill='black', fontSize=12).encode(
    text='code:O'
)

In [15]:
text

alt.Chart(...)

In [16]:
chart = (heatmap + text).properties(width=350, height=250).configure_view(strokeOpacity=0)

In [17]:
# display the chart
chart

alt.LayerChart(...)

Introduce `magma` color scheme 

Light gray background must be introduced later because I cannot combine Objects with a 'config' attribute.
This will be introduced later:

```
.configure_view(
     stroke=None,
     fill='#d9d9d9'
)
```


In [18]:
heatmap = alt.Chart(df).mark_rect(stroke=None, cornerRadius=5).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None),
    color=alt.Color(
        'days:Q', 
        scale=alt.Scale(
            type='log', 
            domain=[0.1, 6000], 
            scheme='magma', 
            reverse=True),
        legend=alt.Legend(
            title='Days (log)',
            gradientLength=200,
            labelFontSize=12,
            titleFontSize=14
        )),
    tooltip=['code', 'days']
).properties(
    width=600,
    height=400,
    title='State-Level Days Heatmap (Log Scale)'
)

In [19]:
heatmap

alt.Chart(...)

Display Chart object again

In [20]:
chart

alt.LayerChart(...)

Replace chart object with combined heatmap and text

In [21]:
chart = (heatmap + text).properties(width=350, height=250) #.configure_view(strokeOpacity=0)

In [22]:
# display the chart
chart

alt.LayerChart(...)

The above is good but I can't see all of the labels so need to make them white with an outline so that they all show up (including HI)

#### Fine-tuning text labels

Including `configure_view` attribute only so that the white text labels can be seen. 

In [23]:
# Text overlay layer
labels = heatmap.mark_text(
    align='center',
    baseline='middle',
    # color='white',
    fill='white',
    fontSize=14,
    fontWeight='bold'
).encode(
    text='code:O'
).configure_view(
     stroke=None,
     fill='#d9d9d9'
)


In [24]:
labels

alt.Chart(...)

Recreating the labels while omitting the `configure_view()` object

In [25]:
labels = heatmap.mark_text(
    align='center',
    baseline='middle',
    # color='white',
    fill='white',
    fontSize=14,
    fontWeight='bold'
).encode(
    text='code:O'
)

In [26]:
# Combine layers
chart = (heatmap + labels).properties(
    width=600,
    height=400,
    title='State-Level Days Heatmap (Reversed Magma + Labels)'
).configure_view(
    stroke=None,
    fill='#d9d9d9'  # Set the background color to light gray
)

chart.display()

alt.LayerChart(...)

In [35]:

# Create a base chart for text layers without color encoding
base_text = alt.Chart(df).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None)
)

# Create fake text outline using small offsets in 4 directions
# offsets = [(-0.5, 0), (0.5, 0), (0, -0.5), (0, 0.5)]
# offsets = [(-1.5, 0), (1.5, 0), (0, 1.5), (0, 1.5)]
offsets = [(-2, 0), (2, 0), (0, -2), (0, 2)]

shadow_layers = [
    base_text.mark_text(
    # heatmap.mark_text(
        align='center',
        baseline='middle',
        fill='black',
        opacity=0.4,
        fontSize=12,
        fontWeight='bold',
        dx=dx,
        dy=dy
    ).encode(
        text='code:O'
    )
    for dx, dy in offsets
]

# Foreground white text layer
text_foreground = base_text.mark_text( #heatmap.mark_text(
    align='center',
    baseline='middle',
    # color='white',
    fill='white',
    fontSize=12,
    # fontWeight='bold'
).encode(
    text='code:O'
)

In [36]:
# Combine all layers
chart = alt.layer(
    heatmap,
    *shadow_layers,
    text_foreground
).properties(
    width=600,
    height=400,
    title="Where I've spent my days"
).configure_view(
    stroke=None,
    fill='#d9d9d9'  # Set the background color to light gray
).resolve_scale(
    color='independent'
)

chart.display()


alt.LayerChart(...)

#### Uses `mark_circle()` method

In [ ]:
heatmap = alt.Chart(df).mark_circle(stroke=None, cornerRadius=5).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None),
    color='id:Q'
)

In [ ]:
heatmap

What is `mark_geoshape()`?

In [ ]:
alt.Chart(df).mark_geoshape()